# BaCP ablation ladder

**Run this notebook top to bottom. It produces the ladder table.**

Every rung changes exactly one thing and inherits everything else from the rung
below it, so a difference between adjacent rows is attributable to that one
thing. The ladder is declared as data in `project/experiments/manifest.py`; this
notebook only drives it.

It is safe to re-run and safe to interrupt. Completion is keyed off the run
records themselves, so a second execution runs only what is missing. To force one
rung to re-run, delete its record from `results/runs/`.

**The only thing you normally change is `TIER` in the next cell.**

| Tier | What it runs | Where |
|---|---|---|
| 0 | every rung, loaders truncated to 2 batches | CPU, minutes. Proves the pipeline. Numbers are meaningless **by construction**. |
| 1 | the spine: G1 gate, Stage A 2x2, Stage C in full, Stage D | GPU. The minimum set that makes the paper defensible. |
| 2 | + Stage B and the backward leave-one-out pass | GPU. Adds the necessity column. |
| 3 | everything at 8 seeds | GPU. |

Tier 0 is not a result and must never be reported as one. Its records are stamped
`is_smoke=True`, the table prints a warning naming them, and
`test_experiment_invariants.py` refuses to let smoke and real rows share a table.

In [ ]:
# --- the one constant -------------------------------------------------------
TIER = 0

# Everything below is setup.
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'project').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'project').exists(), f'could not find the repo root from {Path.cwd()}'

for p in (ROOT / 'project', ROOT / 'project' / 'experiments', ROOT / 'project' / 'scripts'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

os.environ['BACP_TIER'] = str(TIER)
os.environ.setdefault('MPLBACKEND', 'Agg')

import ladder, manifest as M, runner            # noqa: E402
import results as R                             # noqa: E402

print('repo   ', ROOT)
print('results', R.results_dir())

## 1. Environment

Recorded onto every run, so a number can always be traced back to the machine
that produced it. If `cuda` is False here and `TIER > 0`, stop -- a tier-1 sweep
on CPU will not finish.

In [ ]:
import json, torch

env = R.capture_environment()
git = R.capture_git()

print(f"torch      {env['torch_version']}")
print(f"cuda       {torch.cuda.is_available()}  {env['gpu_name'] or ''}"
      f"{' ' + str(env['gpu_mem_total_gb']) + ' GB' if env['gpu_mem_total_gb'] else ''}")
print(f"git        {git['git_sha'][:12] if git['git_sha'] else '?'} "
      f"({git['git_branch']}){' DIRTY: ' + str(git['git_n_dirty']) + ' file(s)' if git['git_dirty'] else ''}")
print(f"fingerprint {R.code_fingerprint()[:16]}")

if TIER > 0 and not torch.cuda.is_available():
    print('\n!! TIER > 0 with no GPU visible. A tier-1 sweep is ~95 full-length '
          'runs; on CPU that is weeks. Set TIER = 0 or attach a GPU.')

## 2. Verify the code before spending anything on it

The unit suite must be green before a sweep starts. This is not ceremony: eight
of the bugs found in this codebase meant something reported in the previous paper
never actually executed -- EAST's masks never updated, the fine-tune optimizer was
never swapped, the SST-2 path was unreachable. A red suite means the numbers below
would describe something other than the method.

Note what this does and does not prove. A green suite shows the code does what it
was written to do. It says nothing about whether BaCP works, and the entire suite
would pass on a method that has no effect. The experiment invariants in section 6
are the layer that protects the *paper*.

In [ ]:
proc = subprocess.run(
    [sys.executable, '-m', 'pytest', '-W', 'ignore::DeprecationWarning',
     '-W', 'ignore::FutureWarning'],
    cwd=str(ROOT), capture_output=True, text=True)
print(proc.stdout[-3000:])

R.register_gate('00_verify', [
    ('unit suite green', proc.returncode == 0,
     proc.stdout.strip().splitlines()[-1] if proc.stdout.strip() else f'rc={proc.returncode}'),
])
assert proc.returncode == 0, 'fix the suite before running a sweep'

## 3. The plan

`validate()` checks the ladder against the live registries: every pruner it names
must exist, every rung must resolve to what it declares, and no two rungs may
resolve identically. A manifest that references a pruner nobody implemented is
worse than no manifest, because it looks like a plan.

`budget()` derives its projection from `s_per_epoch_mean` in the existing records
rather than from an assumed throughput, so once tier 0 has run once the GPU-hour
estimate is arithmetic on measured timings.

In [ ]:
problems = M.validate(strict=False)
assert not problems, problems

print(M.summary(TIER))
print()
print(json.dumps(M.budget(TIER), indent=2, default=str))

path = M.write_manifest_csv(tier=TIER)
print(f'\nintended grid -> {path}')

### What each rung is for

Worth reading once before spending GPU on it. The forward direction is primary:
it localises where accuracy starts to be lost on the way up to BaCP, which a
leave-one-out star around the full method cannot do.

In [ ]:
for r in M.LADDER:
    if r.id not in M.TIERS[TIER]['rungs']:
        continue
    gate = f"  [GATE {r.gate}]" if r.gate else ''
    print(f'{r.id:<10} {r.label}{gate}')
    print(f'{"":10} changes: {r.changes}')
    if r.note:
        print(f'{"":10} {r.note}')
    print()

## 4. Run

Each cell is its own subprocess. That is not fastidiousness: `pruning_factory`
keeps the prunable scope in module-level globals, so running twenty configs in one
kernel means config 7 can silently inherit config 6's scope -- and every sparsity
number depends on it, so the corruption would be invisible and total.

A crashed cell writes a `status='failed'` record rather than nothing, so a failure
is a row in the table rather than a silence.

Set `DRY_RUN = True` first to see the exact command lines without executing them.

In [ ]:
DRY_RUN = False

summary = runner.run_grid(TIER, dry_run=DRY_RUN, resume=True)
print()
print(json.dumps({k: v for k, v in summary.items() if k != 'results'}, indent=2))

## 5. The ladder table

`delta` is against the rung this one inherits from. Stage A is unpaired (the mask
lottery dominates, so seeds do not meaningfully pair across arms); Stages B-D are
seed-paired.

The **noise floor** printed under the table is the median within-rung standard
deviation. Nothing below it is bolded and nothing below it is a result. The
previous paper's ablation deltas were all under 0.4 points with no error bars at
all, which is how a table claims something it cannot support.

In [ ]:
out = ladder.report(TIER)
agg = out['agg']

### Gates

These are hard stops, in order. Each halts the ladder and triggers a re-think of
the method rather than a write-up with a caveat -- and they are placed so that a
stop arrives after 0-11 GPU-hours rather than after the whole sweep.

| Gate | Fires when | Meaning |
|---|---|---|
| **G1** | C-null differs from C0 by more than 0.5pp | The BaCP code path does not reproduce plain CE. Every downstream number would measure that discrepancy rather than the method. **Implementation bug, not a result.** |
| **G3** | neither KD nor feature-KD can reach +0.5pp | No dense-teacher signal helps in this regime, so BaCP has no mechanism to exploit. |
| **G4** | C3 does not beat C2 | The contrastive *form* is indistinguishable from plain feature distillation. This is the paper's central claim. |
| **G5** | both +SnC and +PrC CIs contain zero | Do not tune a method whose own components are indistinguishable from each other. |

In [ ]:
stops = [g for g in out['gates'] if g[1] == 'STOP']
pending = [g for g in out['gates'] if g[1] == 'pending']

if stops:
    print('HARD STOP:')
    for name, _, detail in stops:
        print(f'  {name}: {detail}')
    print('\nThe ladder halts here by design. The next stage would spend GPU-hours '
          'measuring something the controls have already ruled out.')
elif pending:
    print(f'{len(pending)} gate(s) still pending -- not enough runs yet:')
    for name, _, detail in pending:
        print(f'  {name}: {detail}')
else:
    print('All gates clear.')

## 6. Experiment invariants

A categorically different layer from the unit suite. These run against the
*records*, not the code, and ask whether the numbers a table would print are
comparable to each other and to the literature: did the run hit the sparsity it
claims, is the sparsity denominator the same in both arms, did the two arms share
a protocol, is the seed recorded, was any of the test set silently dropped.

No unit test can substitute for these, because none of these questions is about
code.

In [ ]:
proc = subprocess.run(
    [sys.executable, '-m', 'pytest', '-m', 'results', '-s',
     '-W', 'ignore::DeprecationWarning'],
    cwd=str(ROOT), capture_output=True, text=True)
print(proc.stdout[-6000:])
if proc.returncode != 0:
    print('\n!! An invariant failed. The table above is not safe to publish until '
          'it is resolved -- these are the assertions that protect the paper.')

## 7. Output for the paper

`table_ladder.tex` is booktabs and self-contained. Its caption prints the noise
floor and the sample sizes, and nothing inside the noise floor is bolded.

The provenance header comments record the git sha, the code fingerprint and the
row counts, so a number in the paper can be traced back to the exact code that
produced it.

In [ ]:
tex = out['paths'].get('tex')
if tex:
    print(Path(tex).read_text(encoding='utf-8'))

In [ ]:
# Per-seed values, for the appendix. With five numbers, print the five numbers --
# a star on an n<=5 cell is theatre.
for r in agg.itertuples():
    if r.n_ok:
        vals = ', '.join(f'{v:.2f}' for v in r.values)
        print(f'{r.rung:<10} seeds {r.seeds} -> {vals}')